## Setup

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [17]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [18]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



items: 28,343 rows x 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.9,12.69
2,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.9,11.85


... (8 outputs left out)


geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [19]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [20]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


In [22]:
print(items.shape)
print(products.shape)
print(orders.shape)
print(customers.shape)
print(sellers.shape)

(28343, 7)
(13344, 9)
(25000, 8)
(25000, 5)
(2234, 4)


In [23]:
print("Items:", items.columns.tolist())
print("Products:", products.columns.tolist())
print("Orders:", orders.columns.tolist())
print("Customers:", customers.columns.tolist())
print("Sellers:", sellers.columns.tolist())

Items: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
Products: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
Orders: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Customers: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Sellers: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [24]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

items.to_sql("items", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")
orders.to_sql("orders", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
sellers.to_sql("sellers", conn, index=False, if_exists="replace")

query = """
SELECT
    i.order_id,
    i.order_item_id,
    i.freight_value,
    i.price,
    p.product_category_name,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,
    c.customer_state,
    c.customer_zip_code_prefix,
    s.seller_state,
    s.seller_zip_code_prefix
FROM items AS i
INNER JOIN products AS p
    ON i.product_id = p.product_id
INNER JOIN orders AS o
    ON i.order_id = o.order_id
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
INNER JOIN sellers AS s
    ON i.seller_id = s.seller_id
"""

freight_abt = pd.read_sql_query(query, conn)

In [25]:
print("Rows and Columns:", freight_abt.shape)
print("Missing Values:")
print(freight_abt.isnull().sum())
print("Duplicate Rows:", freight_abt.duplicated().sum())

Rows and Columns: (28343, 13)
Missing Values:
order_id                      0
order_item_id                 0
freight_value                 0
price                         0
product_category_name       424
product_weight_g              2
product_length_cm             2
product_height_cm             2
product_width_cm              2
customer_state                0
customer_zip_code_prefix      0
seller_state                  0
seller_zip_code_prefix        0
dtype: int64
Duplicate Rows: 0


In [26]:
freight_abt_clean = freight_abt.dropna().copy()

print("Original Rows:", len(freight_abt))
print("Clean Rows:", len(freight_abt_clean))
print("Rows Removed:", len(freight_abt) - len(freight_abt_clean))
print("Remaining Missing Values:", freight_abt_clean.isnull().sum().sum())

Original Rows: 28343
Clean Rows: 27919
Rows Removed: 424
Remaining Missing Values: 0


In [27]:
columns = [
    "freight_value",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print((freight_abt_clean[columns] <= 0).sum())

freight_value        108
product_weight_g       0
product_length_cm      0
product_height_cm      0
product_width_cm       0
dtype: int64


In [28]:
print("Zero Freight:", (freight_abt_clean["freight_value"] == 0).sum())
print("Negative Freight:", (freight_abt_clean["freight_value"] < 0).sum())

print(freight_abt_clean["freight_value"].describe())

Zero Freight: 108
Negative Freight: 0
count    27919.000000
mean        19.997851
std         15.415460
min          0.000000
25%         13.120000
50%         16.350000
75%         21.280000
max        339.590000
Name: freight_value, dtype: float64


In [29]:
from sklearn.model_selection import GroupShuffleSplit

# First split: 70% training, 30% temporary
split1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(
        freight_abt_clean,
        groups=freight_abt_clean["order_id"]
    )
)

train = freight_abt_clean.iloc[train_idx]
temp = freight_abt_clean.iloc[temp_idx]

# Second split: temporary data into validation and test
split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    split2.split(temp, groups=temp["order_id"])
)

validation = temp.iloc[val_idx]
test = temp.iloc[test_idx]

print("Training Rows:", len(train))
print("Validation Rows:", len(validation))
print("Testing Rows:", len(test))
print("Total Rows:", len(train) + len(validation) + len(test))

Training Rows: 19627
Validation Rows: 4158
Testing Rows: 4134
Total Rows: 27919


In [30]:
features = [
    "price",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "customer_state",
    "seller_state"
]

target = "freight_value"

X_train = train[features]
y_train = train[target]

X_val = validation[features]
y_val = validation[target]

X_test = test[features]
y_test = test[target]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (19627, 7)
Validation: (4158, 7)
Testing: (4134, 7)


In [31]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Identify numerical and categorical features
numeric_features = [
    "price",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

categorical_features = [
    "customer_state",
    "seller_state"
]

# Prepare the data
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Build Model A
model_a = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])

# Train the model
model_a.fit(X_train, y_train)

print("Model A: Linear Regression trained successfully!")

Model A: Linear Regression trained successfully!


In [32]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Predict freight costs using validation data
y_val_pred = model_a.predict(X_val)

# Calculate evaluation metrics
rmse_a = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae_a = mean_absolute_error(y_val, y_val_pred)
r2_a = r2_score(y_val, y_val_pred)

print("Model A: Linear Regression")
print("Validation RMSE:", round(rmse_a, 2))
print("Validation MAE:", round(mae_a, 2))
print("Validation R²:", round(r2_a, 4))

Model A: Linear Regression
Validation RMSE: 10.3
Validation MAE: 5.37
Validation R²: 0.5418


In [34]:
# Predict freight costs using the test data
y_test_pred_a = model_a.predict(X_test)

# Calculate test metrics
rmse_test_a = np.sqrt(mean_squared_error(y_test, y_test_pred_a))
mae_test_a = mean_absolute_error(y_test, y_test_pred_a)
r2_test_a = r2_score(y_test, y_test_pred_a)

print("Model A: Linear Regression - Test Results")
print("Test RMSE:", round(rmse_test_a, 2))
print("Test MAE:", round(mae_test_a, 2))
print("Test R²:", round(r2_test_a, 4))

Model A: Linear Regression - Test Results
Test RMSE: 10.35
Test MAE: 5.68
Test R²: 0.5715


In [35]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Build Model B
model_b = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regression", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Train Model B
model_b.fit(X_train, y_train)

print("Model B: Random Forest trained successfully!")

Model B: Random Forest trained successfully!


In [36]:
# Predict freight costs using validation data
y_val_pred_b = model_b.predict(X_val)

# Calculate validation metrics
rmse_b = np.sqrt(mean_squared_error(y_val, y_val_pred_b))
mae_b = mean_absolute_error(y_val, y_val_pred_b)
r2_b = r2_score(y_val, y_val_pred_b)

print("Model B: Random Forest Regression")
print("Validation RMSE:", round(rmse_b, 2))
print("Validation MAE:", round(mae_b, 2))
print("Validation R²:", round(r2_b, 4))

Model B: Random Forest Regression
Validation RMSE: 9.11
Validation MAE: 4.69
Validation R²: 0.6412


In [37]:
# Predict freight costs using the test data
y_test_pred_b = model_b.predict(X_test)

# Calculate test metrics
rmse_test_b = np.sqrt(mean_squared_error(y_test, y_test_pred_b))
mae_test_b = mean_absolute_error(y_test, y_test_pred_b)
r2_test_b = r2_score(y_test, y_test_pred_b)

print("Model B: Random Forest - Test Results")
print("Test RMSE:", round(rmse_test_b, 2))
print("Test MAE:", round(mae_test_b, 2))
print("Test R²:", round(r2_test_b, 4))

Model B: Random Forest - Test Results
Test RMSE: 9.9
Test MAE: 4.95
Test R²: 0.608
